# 01 - Deploy the shared Search environment

Bicep creates one dedicated resource group, one paid Basic Search service (one replica/partition, North Europe), and two user role assignments. No indexes, documents, queries, models, or private networking are created.

Run from this notebook's directory using the setup kernel. Complete the [README prerequisites](../README.md#prerequisites-and-approval) first. Deployment, provider registration, and cleanup each require explicit approval. This notebook never registers providers or changes policy automatically.

## Step 1 - Resolve the selected subscription and setup user

In [ ]:
import json
import os
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

from azure.identity import AzureCliCredential
from azure.mgmt.resource.resources import ResourceManagementClient
from azure.mgmt.resource.deployments import DeploymentsMgmtClient

demo_root = Path('..').resolve()
sys.path.insert(0, str(demo_root))
from app.notebook_helpers import environment_from_outputs, resolve_az_cli

RESOURCE_GROUP = "rg-search02-hybrid-pagination-ne"
LOCATION = 'northeurope'
DEMO_ID = 'search02-hybrid-pagination'
DEPLOYMENT_NAME = 'search02-hybrid-pagination'
generated = demo_root / 'outputs' / 'generated'
generated.mkdir(parents=True, exist_ok=True)

az_cmd = resolve_az_cli()
if az_cmd is None:
    raise RuntimeError('Azure CLI not found. Complete setup first.')
result = subprocess.run([az_cmd, 'account', 'show', '--output', 'json'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(f'az account show failed ({result.returncode})\nstderr: {result.stderr.strip()}\nstdout: {result.stdout.strip()}')
account = json.loads(result.stdout)
subscription_id = os.environ.get('AZURE_SUBSCRIPTION_ID', account['id'])
if subscription_id.casefold() != account['id'].casefold():
    result = subprocess.run([az_cmd, 'account', 'set', '--subscription', subscription_id], capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f'az account set failed ({result.returncode})\nstderr: {result.stderr.strip()}\nstdout: {result.stdout.strip()}')
    result = subprocess.run([az_cmd, 'account', 'show', '--output', 'json'], capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f'az account show failed ({result.returncode})\nstderr: {result.stderr.strip()}\nstdout: {result.stdout.strip()}')
    account = json.loads(result.stdout)
    if subscription_id.casefold() != account['id'].casefold():
        raise RuntimeError(f'Azure CLI did not select the requested subscription: {subscription_id}')
    print(f'Azure CLI active subscription changed to {subscription_id}.')
if account['user']['type'].casefold() != 'user':
    raise ValueError('Sign in as the setup user, not a service principal or managed identity.')
tenant_id = account['tenantId']
result = subprocess.run([az_cmd, 'ad', 'signed-in-user', 'show', '--query', 'id', '--output', 'tsv'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(f'az ad signed-in-user show failed ({result.returncode})\nstderr: {result.stderr.strip()}\nstdout: {result.stdout.strip()}')
operator_object_id = result.stdout.strip()
if not operator_object_id:
    raise ValueError('The signed-in setup user has no resolved object ID.')
credential = AzureCliCredential(subscription=subscription_id)
resource_client = ResourceManagementClient(credential, subscription_id)
deployments_client = DeploymentsMgmtClient(credential, subscription_id)
print(json.dumps({'subscriptionId': subscription_id, 'tenantId': tenant_id, 'operatorObjectId': operator_object_id, 'resourceGroupName': RESOURCE_GROUP, 'location': LOCATION}, indent=2))

## Step 2 - Confirm current policy, cost, permissions, and ownership

Azure deployment validation and what-if in **Step 4** check the selected template and scope before deployment. No manually authored policy-review file is required. These checks are not a full audit of advisory policies; any underlying policy or permission error must be resolved rather than bypassed.

Review current Basic pricing/capacity and resource-write/role-assignment privileges. If Microsoft.Search is unregistered, obtain separate approval and run the README registration command; then rerun this step. This step checks provider state and ownership without a confirmation flag; it does not deploy resources.

In [ ]:
provider = resource_client.providers.get('Microsoft.Search')
if provider.registration_state != 'Registered':
    raise RuntimeError('Microsoft.Search must be registered with separate approval; see the README command.')
if resource_client.resource_groups.check_existence(RESOURCE_GROUP):
    group = resource_client.resource_groups.get(RESOURCE_GROUP)
    if group.tags is None or group.tags.get('demo') != DEMO_ID:
        raise ValueError('Existing group is unrelated or unowned; it must not be adopted.')
print('Provider registration and ownership boundary checked. No deployment submitted.')

## Step 3 - Compile Bicep locally

In [ ]:
compiled_path = generated / 'main.json'
result = subprocess.run([az_cmd, 'bicep', 'build', '--file', str(demo_root / 'infra' / 'main.bicep'), '--outfile', str(compiled_path)], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(f'az bicep build failed ({result.returncode})\nstderr: {result.stderr.strip()}\nstdout: {result.stdout.strip()}')
print(result.stderr.strip())
template = json.loads(compiled_path.read_text(encoding='utf-8'))
deployment_parameters = {'location': LOCATION, 'properties': {'mode': 'Incremental', 'template': template, 'parameters': {'resourceGroupName': {'value': RESOURCE_GROUP}, 'location': {'value': LOCATION}, 'operatorObjectId': {'value': operator_object_id}, 'demoId': {'value': DEMO_ID}}}}
print(f'Compiled {compiled_path.name}; Bicep remains the resource source of truth.')

## Step 4 - Validate and review what-if

These SDK requests validate/preview the compiled template in the selected subscription, including applicable enforced deployment policies; they do not apply the resource deployment. No separate policy-review file is needed. This is not a full audit of advisory policies or a guarantee that deployment will succeed. Stop on any validation error, unrelated resource, or unapproved setting.

In [ ]:
validation = deployments_client.deployments.begin_validate_at_subscription_scope(DEPLOYMENT_NAME, deployment_parameters).result()
if validation.error is not None:
    raise RuntimeError(str(validation.error))
preview = deployments_client.deployments.begin_what_if_at_subscription_scope(DEPLOYMENT_NAME, deployment_parameters).result()
if preview.error is not None:
    raise RuntimeError(str(preview.error))
(generated / 'what-if.json').write_text(json.dumps(preview.as_dict(), indent=2), encoding='utf-8')
print(json.dumps(preview.as_dict(), indent=2))

## Step 5 - Explicitly deploy the paid environment

Running this step submits the paid deployment immediately after a successful what-if; there is no confirmation flag. Review the selected subscription and preview first. The service costs money while idle. Re-running this step with unchanged inputs must preserve the same group, service, and assignment identities.

In [ ]:
if preview.status != 'Succeeded':
    raise RuntimeError(f'What-if did not succeed: {preview.status}')
deployment = deployments_client.deployments.begin_create_or_update_at_subscription_scope(DEPLOYMENT_NAME, deployment_parameters).result()
if deployment.properties.provisioning_state != 'Succeeded':
    raise RuntimeError(f'Deployment did not succeed: {deployment.properties.provisioning_state}')
print(f'Deployment completed: {deployment.name}. Export and readiness checks are still required.')

## Step 6 - Export the non-secret handoff

Require the actual completed deployment outputs. A missing or inconsistent value fails rather than reusing an old environment file. The same nine values serve both future clients.

In [ ]:
if deployment.properties.provisioning_state != 'Succeeded':
    raise RuntimeError('Export requires the actual successful deployment result.')
environment = environment_from_outputs(deployment.properties.outputs, subscription_id=subscription_id, tenant_id=tenant_id, resource_group=RESOURCE_GROUP, operator_object_id=operator_object_id)
env_path = demo_root / 'env' / '.env'
env_path.write_text(''.join(f'{key}={value}\n' for key, value in environment.items()), encoding='utf-8')
os.environ.update(environment)
deployment_evidence = {'observedAt': datetime.now(timezone.utc).isoformat(), 'deploymentName': DEPLOYMENT_NAME, 'provisioningState': deployment.properties.provisioning_state, 'environment': environment, 'metadataStatus': 'not-run'}
(generated / 'deployment.json').write_text(json.dumps(deployment_evidence, indent=2), encoding='utf-8')
print(f'Wrote all {len(environment)} non-secret values to {env_path}. Continue to 02_configure.ipynb, then 03_test.ipynb.')